# 03. Classical Baselines: Majority Category Baseline

Fundamentals of Natural Language / NLP-I, Universitat Autonoma de Barcelona, academic year 2025-2026.

Team 10: Phoebe Iglesias (1713459), David Redrejo (1790336), and Pau Rossell (1750424). Supervisors: Ernest Valveny and Lei Kang.

In this notebook we start the modeling phase with the simplest possible model: always predict the most frequent ICD-10 category prefix in the training split. This model is not designed to win the Kaggle competition. It is a sanity baseline that tells us how much of the validation accuracy can be explained by class imbalance alone.

## Why This Baseline Is Necessary

From the EDA we already saw that the category distribution is imbalanced. After reading the ICD coding survey, we also understood that imbalance is a central problem in automated ICD coding: frequent clinical categories dominate the data, while rare categories are still medically important.

A majority baseline gives us the first honest threshold. If a later model cannot beat it clearly, the model is probably learning the dataset prior rather than useful linguistic or clinical evidence from the literal. Beating this model is therefore the minimum requirement for any meaningful approach.

## What v00 Does

The script `models/v00_majority_baseline.py` follows the shared model-version contract:

1. Load `data/processed/train_required_clean.csv` and `data/processed/leaderboard_required_clean.csv`.
2. Use the existing annotation contract where `y_category` is derived from the first character of `Code`.
3. Create an 80/20 validation split stratified by `label_id` with seed 42.
4. Find the most frequent `y_category` in the training split.
5. Predict that category for every validation and leaderboard example.
6. Save metrics, validation predictions, a Kaggle submission, and a run summary.

In [ ]:
from pathlib import Path
import json
import pandas as pd

metrics_path = Path('../outputs/metrics/v00_majority_baseline_metrics.json')
submission_path = Path('../submissions/v00_majority_baseline_submission.csv')
val_predictions_path = Path('../outputs/predictions/v00_majority_baseline_val_predictions.csv')

metrics = json.loads(metrics_path.read_text())
submission = pd.read_csv(submission_path)
val_predictions = pd.read_csv(val_predictions_path)

results = pd.DataFrame([
    {
        'model': 'v00_majority_baseline',
        'validation_accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'weighted_f1': metrics['weighted_f1'],
        'predicted_category': val_predictions['y_pred'].mode()[0],
        'submission_rows': len(submission),
        'submission_columns': ', '.join(submission.columns),
    }
])
results

## Results

| model | validation accuracy | macro F1 | weighted F1 | predicted category |
|---|---:|---:|---:|---|
| v00_majority_baseline | 0.1252 | 0.0062 | 0.0279 | Z |

The majority category in the training split was `Z`. The model predicted `Z` for every validation example and every leaderboard example. This gives an accuracy of about 12.5%, but macro F1 is almost zero because all non-`Z` classes receive no positive predictions.

This is exactly why the baseline is useful: it shows that class imbalance alone gives a non-trivial accuracy, while also showing that such a model is clinically and linguistically empty.

In [ ]:
assert submission.columns.tolist() == ['id', 'y_category']
assert len(submission) == 6667
submission.head()

## Interpretation for the Next Step

Our next classical baselines must beat 12.5% validation accuracy and, more importantly, must improve macro F1 by predicting more than one category. TF-IDF character n-grams are the next reasonable step because clinical literals contain compact morphology, abbreviations, punctuation, digits, and partial code-like patterns that may be informative even before using RoBERTa.

For the report, this result will be presented as the lower bound of the modeling section: a model that knows only the class prior and ignores the literal text.

## v01: Character TF-IDF + Logistic Regression

After the majority baseline, we moved to the first model that actually reads the clinical literal. Character n-grams are a strong non-deep-learning baseline for this task because the inputs are short, abbreviated, and sometimes inconsistent. Instead of depending only on complete words, the model can learn fragments such as suffixes, prefixes, digits, punctuation patterns, and pieces of Spanish medical terminology.

This connects directly to Basic Text Processing, regex/text-pattern thinking, n-grams, vector-space representations, and the traditional machine learning stage described in the ICD coding survey. It is also a feature-engineering example from Fundamentals of Machine Learning: before learning a neural representation, we manually choose a representation that we believe matches the structure of the data.

## Internal Grid

The script `models/v01_tfidf_char_logreg.py` runs a small internal grid over character n-gram ranges `(2,4)`, `(3,5)`, and `(2,6)`, with `class_weight=None` and `class_weight="balanced"`. We also included a lowercase ablation, but kept required-clean text as the default unless the evidence supported changing it.

In [ ]:
from pathlib import Path
import json
import pandas as pd

grid = pd.read_csv('../reports/tables/v01_tfidf_char_grid.csv')
metrics = json.loads(Path('../outputs/metrics/v01_tfidf_char_logreg_metrics.json').read_text())
grid.head(8)

## v01 Results

| model | validation accuracy | macro F1 | weighted F1 | best preprocessing | best n-grams | class weight |
|---|---:|---:|---:|---|---|---|
| v01_tfidf_char_logreg | 0.5226 | 0.4026 | 0.4949 | required-clean | (3,5) | none |

This model clearly beats the majority baseline. Accuracy rises from about 12.5% to 52.3%, and macro F1 rises from almost zero to 0.4026. This means the model is not only exploiting class imbalance; it is using textual evidence from the literals.

Lowercasing tied the required-clean setting but did not improve it. Because the RoBERTa pipeline should preserve case and because there is no validation gain here, we keep required-clean preprocessing as the default.

In [ ]:
top_ngrams = pd.read_csv('../reports/tables/v01_tfidf_char_top_ngrams.csv')
top_ngrams.head(20)

## What Character N-Grams Capture and Where They Fail

The feature interpretation table shows the highest positive character n-gram coefficients per category. These are not clinical explanations in the strict sense, but they help us see that the model is learning recurring fragments of terms, abbreviations, endings, and short patterns.

The limitation is that character TF-IDF remains shallow. It does not understand clinical context, synonymy, negation, or why two similar literals might belong to different ICD categories. This is why it is a strong classical baseline but not the end of the project. The next models should test whether word-level baselines and later biomedical-clinical RoBERTa can add semantic generalization beyond surface form matching.

## v02: Word TF-IDF + Linear SVM / Logistic Regression

The second classical baseline asks a different question from the character n-gram model: do complete words and short word phrases already contain enough lexical signal to identify the ICD category? This is useful because word-level TF-IDF is closer to the standard vector-space representation introduced in traditional NLP and Machine Learning.

Word n-grams can capture interpretable lexical units such as clinical terms, anatomical words, procedure names, and short collocations. They may work better when the literal contains stable terminology. However, they miss many things that character n-grams handle naturally: morphology inside words, spelling variants, compact abbreviations, punctuation-attached fragments, and rare tokens that do not repeat often.

## v02 Grid

The script `models/v02_tfidf_word_svm.py` tests word TF-IDF with n-gram ranges `(1,1)`, `(1,2)`, and `(1,3)`. It compares LinearSVC and LogisticRegression, with `min_df` values 1 and 2 and `sublinear_tf=True`. The split is the same stratified 80/20 split used by the previous model versions.

In [ ]:
word_grid = pd.read_csv('../reports/tables/v02_tfidf_word_grid.csv')
word_grid.head(8)

## Comparing v00, v01, and v02

The best word-level configuration was unigram TF-IDF with LinearSVC, `min_df=1`, and `sublinear_tf=True`. It is slightly below the character n-gram model in validation accuracy, but it improves macro F1 and weighted F1. This is important for ICD-style tasks because macro F1 is more sensitive to minority classes than plain accuracy.

In [ ]:
comparison = pd.read_csv('../reports/tables/classical_baseline_comparison.csv')
comparison

![Classical baseline comparison](../reports/figures/fig_10_classical_baseline_comparison.png)

| model | validation accuracy | macro F1 | weighted F1 |
|---|---:|---:|---:|
| v00_majority_baseline | 0.1252 | 0.0062 | 0.0279 |
| v01_tfidf_char_logreg | 0.5226 | 0.4026 | 0.4949 |
| v02_tfidf_word_svm | 0.5201 | 0.4742 | 0.5140 |

This comparison gives us a clear checkpoint before moving to Transformers. Sparse vector-space models already learn a lot from the literals, so RoBERTa should not merely beat the majority baseline. It should aim to beat strong TF-IDF baselines and improve generalization beyond surface lexical matching.